# Smoothed daily-rainfall animation

This notebook builds an animated map of consensus daily rainfall over the UK,
**smoothed by interpolating between consecutive days**. Instead of one abrupt
frame per day, several intermediate frames are drawn between each day and the
next (linear interpolation per station, missing values treated as zero), so the
rainfall field appears to evolve continuously.

Each frame is the same map produced by `plot_daily_rainfall_map.py` (median over
the 5 ensemble members, only located stations for the matching `matched_year`),
so the animation shares the static/interactive maps' styling: tall UK framing,
`YlGnBu` square-root colour scale, values in **inches**, coastlines and borders.

Rendering a full multi-year run is thousands of frames, so the work is done in
parallel on the SPICE cluster with SLURM and only the final MP4 is brought back
here for display.

## How the pipeline works

The animation is produced by four SLURM stages, chained with
`--dependency=afterok` so each starts only when the previous one succeeds:

| Stage | Script | SLURM | Purpose |
|-------|--------|-------|---------|
| precompute | `scripts/slurm/render_precompute.sbatch` | 1 job | write `manifest.json` describing every frame and the shard boundaries |
| render | `scripts/slurm/render_array.sbatch` | array `0-(N-1)` | render each shard's contiguous frame range in parallel |
| validate | `scripts/slurm/render_validate.sbatch` | 1 job | confirm every expected `frame_NNNNNNN.png` exists |
| encode | `scripts/slurm/render_encode.sbatch` | 1 job | `ffmpeg` the frame sequence into an H.264 MP4 |

**Deterministic frame indexing.** Every frame has a global index `0 .. total-1`
fixed by the date range and `frames_per_day`. Each render shard owns a
contiguous block of indices, so shards never collide and a failed shard can be
re-run on its own without touching the others.

**Interpolation density.** `RENDER_FRAMES_PER_DAY` is the number of frames
covering each day→next-day step (the default `6` means the source day plus 5
interpolated in-between frames). Raise it for a smoother, longer video.

**Where things are stored** (all under `$PDIR/animation/`):

- `manifest.json` — the run description read by every stage
- `frames/frame_NNNNNNN.png` — the rendered frames (staged via node-local
  `$TMPDIR`, then published to shared disc)
- `rainfall_<start>_<end>.mp4` (+ `.mp4.json` metadata sidecar) — the final video

Everything (date range, interpolation density, shard count, fps, colour scale
and per-stage cluster resources) is configured in `scripts/slurm/config.sh` via
the `RENDER_*` variables, and can be overridden at submit time as shown below.

In [2]:
# Setup: locate the repository root and the SLURM helper scripts.

import os
import subprocess
from pathlib import Path

import src.rainfall_rescue_sqlite as _pkg

repo_root = Path(_pkg.__file__).resolve().parents[2]
slurm_dir = repo_root / "scripts" / "slurm"
submit_script = slurm_dir / "submit_animation.sh"

pdir = Path(os.environ["PDIR"])
animation_dir = pdir / "animation"

print(f"repo_root:      {repo_root}")
print(f"submit script:  {submit_script}")
print(f"animation dir:  {animation_dir}")

repo_root:      /home/users/philip.brohan/Projects/Auto-Daily-Rainfall-QC-MO
submit script:  /home/users/philip.brohan/Projects/Auto-Daily-Rainfall-QC-MO/scripts/slurm/submit_animation.sh
animation dir:  /data/scratch/philip.brohan/ADRQ/animation


## Submit a test run (a single year)

Start with one year (1931) to check the whole pipeline end-to-end before
committing cluster time to the full record. The environment variables below
override the defaults in `config.sh` for this submission only.

- `RENDER_DATE_START` / `RENDER_DATE_END` — the calendar range to animate
- `RENDER_FRAMES_PER_DAY` — interpolation density (source day + in-between frames)
- `RENDER_NUM_SHARDS` — how many render array tasks to split the work across
- `RENDER_FPS` — playback frame rate of the MP4

The cell prints the four submitted job IDs.

In [3]:
# Submit the test-year animation pipeline to SLURM.

test_env = {
    **os.environ,
    "RENDER_DATE_START": "1931-01-01",
    "RENDER_DATE_END": "1931-12-31",
    "RENDER_FRAMES_PER_DAY": "3",   # source day + 2 interpolated frames
    "RENDER_NUM_SHARDS": "50",
    "RENDER_FPS": "30",
}

result = subprocess.run(
    ["bash", str(submit_script)],
    env=test_env,
    cwd=str(repo_root),
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise SystemExit(f"submit_animation.sh failed with code {result.returncode}")

Submitted precompute job: 28540935
Submitted render array: 28540936 (0-49)
Submitted validate job: 28540937
Submitted encode job: 28540938

Animation pipeline submitted. Track with:  squeue -u $USER
Final video lands in: /data/scratch/philip.brohan/ADRQ/animation/ (see manifest output_path)



## Monitor the jobs

Re-run the cell below to watch progress. The pipeline is done when the queue is
empty and the MP4 appears in `$PDIR/animation/`. Per-job logs land in
`$PDIR/slurm_logs/`.

In [28]:
# Show this user's queued/running jobs and any finished videos.

queue = subprocess.run(
    ["squeue", "-u", os.environ["USER"], "-o", "%.18i %.20j %.8T %.10M %.6D %R"],
    capture_output=True,
    text=True,
)
print(queue.stdout or "(squeue produced no output)")

print("\nRendered videos in", animation_dir, ":")
if animation_dir.exists():
    videos = sorted(animation_dir.glob("*.mp4"))
    for v in videos:
        print(f"  {v.name}  ({v.stat().st_size / 1e6:.1f} MB)")
    if not videos:
        n_frames = len(list((animation_dir / "frames").glob("*.png"))) if (animation_dir / "frames").exists() else 0
        print(f"  (no MP4 yet; {n_frames} frames rendered so far)")
else:
    print("  (animation directory does not exist yet)")

             JOBID                 NAME    STATE       TIME  NODES NODELIST(REASON)
            592443                 find  PENDING       0:00      1 (BeginTime)


Rendered videos in /data/scratch/philip.brohan/ADRQ/animation :
  rainfall_1860-01-01_1960-12-31.mp4  (628.0 MB)
  rainfall_1900-01-01_1939-12-31.mp4  (148.2 MB)
  rainfall_1930-01-01_1931-12-31.mp4  (10.1 MB)
  rainfall_1931-01-01_1931-12-31.mp4  (8.4 MB)


## Display the finished animation

Once the encode stage has completed, the cell below embeds the MP4 inline. It
picks the most recently modified video in the animation directory.

In [ ]:
# Embed the most recent rendered animation.

from IPython.display import Video, display

videos = sorted(animation_dir.glob("*.mp4"), key=lambda p: p.stat().st_mtime)
if not videos:
    raise FileNotFoundError(
        f"No MP4 found in {animation_dir}. Has the encode stage finished?"
    )

latest_video = videos[-1]
print(f"Displaying: {latest_video}")
display(Video(str(latest_video), embed=True, width=500))

## Run the full record

When the test year looks right, animate the full available date range. This is a
much larger job — raise `RENDER_NUM_SHARDS` so the frames render in parallel
across many array tasks. Set `RENDER_DATE_START` / `RENDER_DATE_END` to the span
you want (the whole matched record, or any sub-period).

The cell is not executed by default — change `submit_full = True` to launch it.

In [26]:
# Submit the full-range animation pipeline (guarded so it does not run by accident).

submit_full = True  # set to True to launch the full-record render

full_env = {
    **os.environ,
    "RENDER_DATE_START": "1860-01-01",
    "RENDER_DATE_END": "1960-12-31",
    "RENDER_FRAMES_PER_DAY": "3",
    "RENDER_NUM_SHARDS": "500",
    "RENDER_FPS": "30",
}

if submit_full:
    result = subprocess.run(
        ["bash", str(submit_script)],
        env=full_env,
        cwd=str(repo_root),
        capture_output=True,
        text=True,
    )
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise SystemExit(f"submit_animation.sh failed with code {result.returncode}")
else:
    print("submit_full is False - not submitting. Set it to True to launch the full render.")

Submitted precompute job: 28548968
Submitted render array: 28548969 (0-499)
Submitted validate job: 28548970
Submitted encode job: 28548971

Animation pipeline submitted. Track with:  squeue -u $USER
Final video lands in: /data/scratch/philip.brohan/ADRQ/animation/ (see manifest output_path)

